In [ ]:
# TODO: fill doc

In [ ]:
# Get PR info: username, location and number of PRs from GitHub

!uv pip install requests

import csv
import os
import time

import requests

GITHUB_TOKEN: str = os.getenv("GITHUB_TOKEN")

HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"token {GITHUB_TOKEN}",
}


def get_all_pr_authors_with_counts(repo: str) -> dict[str, int]:
    print("Fetching merged PR authors (excluding bots)...")
    author_counts: dict[str, int] = {}
    page: int = 1
    per_page: int = 100

    while True:
        url = f"https://api.github.com/repos/{repo}/pulls"
        params = {"state": "closed", "per_page": per_page, "page": page}
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code != 200:
            print(f"Failed to fetch PRs: {response.status_code}, {response.text}")
            break

        prs = response.json()
        if not prs:
            break

        for pr in prs:
            if not pr.get("merged_at"):
                continue

            user = pr.get("user")
            username = user.get("login", "") if user else ""
            if not username or not user:
                raise ValueError(f"Cannot get username for {user}")

            if username and "[bot]" not in username:
                author_counts[username] = author_counts.get(username, 0) + 1

        print(
            f"Fetched page {page}, total unique merged authors so far: {len(author_counts)}"
        )
        page += 1

    return author_counts


def get_user_location(username: str) -> str | None:
    url = f"https://api.github.com/users/{username}"
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        data = response.json()
        return data.get("location")
    else:
        print(f"Failed to fetch user {username}: {response.status_code}")
        return None


repo: str = "materialsproject/pymatgen"
author_counts = get_all_pr_authors_with_counts(repo)

print("\nFetching PR info...")
user_locations: list[dict[str, str | int]] = []
for i, (username, count) in enumerate(sorted(author_counts.items())):
    location = get_user_location(username)
    user_locations.append(
        {"username": username, "location": location, "pr_count": count}
    )
    print(f"{i + 1:3d}/{len(author_counts)} {username}: {location} ({count} PRs)")
    time.sleep(1)

# Save to CSV
with open("pr_info.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["username", "location", "pr_count"])
    writer.writeheader()
    writer.writerows(user_locations)

print("Saved pr_info.csv")

Fetching merged PR authors (excluding bots)...
Fetched page 1, total unique merged authors so far: 27
Fetched page 2, total unique merged authors so far: 41
Fetched page 3, total unique merged authors so far: 57
Fetched page 4, total unique merged authors so far: 67
Fetched page 5, total unique merged authors so far: 77
Fetched page 6, total unique merged authors so far: 86
Fetched page 7, total unique merged authors so far: 94
Fetched page 8, total unique merged authors so far: 101
Fetched page 9, total unique merged authors so far: 109
Fetched page 10, total unique merged authors so far: 116
Fetched page 11, total unique merged authors so far: 118
Fetched page 12, total unique merged authors so far: 126
Fetched page 13, total unique merged authors so far: 135
Fetched page 14, total unique merged authors so far: 144
Fetched page 15, total unique merged authors so far: 159
Fetched page 16, total unique merged authors so far: 174
Fetched page 17, total unique merged authors so far: 187
